Below encoder decoder is for left black vertical line/stack

In [ ]:
import cv2
import numpy as np

INPUT_VIDEO = "/content/4ktest.mp4"
OUTPUT_VIDEO = "/content/encoded_black_left.mp4"

# CHANGE MESSAGE HERE ONLY
MESSAGE = [4,3,5,1,5,5,1,2]   # example: 43515512

DIGIT_TO_SYMBOL = {
    1: "+",
    2: "-",
    3: "x",
    4: "|",
    5: "--",
    6: "||"
}

BLACK = (0, 0, 0)
ROI = 200
THICK = 18

INTERVAL_SEC = 5
ON_SEC = 1

def get_vertical_left_pos(h, index):
    x = int(0.12 * W)                  # left side
    y_start = int(0.2 * h)             # start a bit down
    gap = ROI + 30
    y = y_start + index * gap
    return y, x

def draw_symbol(sym):
    img = np.zeros((ROI, ROI, 3), dtype=np.uint8)
    img[:] = 255  # white background

    c = ROI // 2
    m = 30

    if sym == "|":
        cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
    elif sym == "||":
        cv2.line(img,(c-25,m),(c-25,ROI-m),BLACK,THICK)
        cv2.line(img,(c+25,m),(c+25,ROI-m),BLACK,THICK)
    elif sym == "-":
        cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
    elif sym == "--":
        cv2.line(img,(m,c-20),(ROI-m,c-20),BLACK,THICK)
        cv2.line(img,(m,c+20),(ROI-m,c+20),BLACK,THICK)
    elif sym == "+":
        cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
        cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
    elif sym == "x":
        cv2.line(img,(m,m),(ROI-m,ROI-m),BLACK,THICK)
        cv2.line(img,(ROI-m,m),(m,ROI-m),BLACK,THICK)

    return img

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(3))
H = int(cap.get(4))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

interval_f = int(INTERVAL_SEC * fps)
on_f = int(ON_SEC * fps)
f = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    slot = (f // interval_f) % len(MESSAGE)

    if f % interval_f < on_f:
        digit = MESSAGE[slot]
        sym = DIGIT_TO_SYMBOL[digit]
        block = draw_symbol(sym)

        cy, cx = get_vertical_left_pos(H, slot)
        y, x = cy - ROI//2, cx - ROI//2

        mask = block[:,:,0] < 250   # black pixels
        frame[y:y+ROI, x:x+ROI][mask] = block[mask]

    out.write(frame)
    f += 1

cap.release()
out.release()

print("Encoded message:", MESSAGE)
print("Saved to:", OUTPUT_VIDEO)


Encoded message: [4, 3, 5, 1, 5, 5, 1, 2]
Saved to: /content/encoded_black_left.mp4


In [ ]:
import cv2
import numpy as np
from collections import Counter

VIDEO = "/content/recorded_encoded_black_left.mp4"

EXPECTED_MESSAGE = [4,3,5,1,5,5,1,2]

DIGIT_TO_SYMBOL = {
    1: "+",
    2: "-",
    3: "x",
    4: "|",
    5: "--",
    6: "||"
}

EXPECTED_OPS = set(DIGIT_TO_SYMBOL[d] for d in EXPECTED_MESSAGE)

# BLACK COLOR RANGE (HSV)
LOW_BLACK  = np.array([0, 0, 0])
HIGH_BLACK = np.array([180, 255, 60])

def classify_operator(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    lines = cv2.HoughLines(edges, 1, np.pi/180, 80)
    if lines is None:
        return None

    angles = [l[0][1] for l in lines]
    vert  = sum(abs(a - np.pi/2) < 0.12 for a in angles)
    horiz = sum(abs(a) < 0.12 for a in angles)
    diag  = sum(abs(a - np.pi/4) < 0.15 or abs(a - 3*np.pi/4) < 0.15 for a in angles)

    if vert and horiz:
        return "+"
    if vert >= 2:
        return "||"
    if horiz >= 2:
        return "--"
    if diag >= 2:
        return "x"
    if vert:
        return "|"
    if horiz:
        return "-"
    return None

cap = cv2.VideoCapture(VIDEO)
detected_ops = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOW_BLACK, HIGH_BLACK)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5),np.uint8))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        if w*h < 1200:
            continue

        roi = frame[y:y+h, x:x+w]
        op = classify_operator(roi)
        if op:
            detected_ops.append(op)

cap.release()

counts = Counter(detected_ops)
print("Detected operator counts:", counts)

missing = EXPECTED_OPS - set(counts.keys())

if not missing:
    print("✅ WATERMARK VERIFIED")
    print("Matched payload:", "".join(map(str, EXPECTED_MESSAGE)))
else:
    print("❌ WATERMARK NOT VERIFIED")
    print("Missing operators:", missing)


Detected operator counts: Counter({'+': 10235, '||': 4694, '|': 1540, '-': 1082, '--': 753, 'x': 222})
✅ WATERMARK VERIFIED
Matched payload: 43515512


In [ ]:
# import cv2
# import numpy as np

# INPUT_VIDEO = "/content/4ktest.mp4"
# OUTPUT_VIDEO = "/content/encoded_15452654651354.mp4"

# # CHANGE MESSAGE HERE ONLY
# MESSAGE = [1,5,4,5,2,6,5,4,6,5,1,3,5,4]   # example: 15452654651354

# DIGIT_TO_SYMBOL = {
#     1: "+",
#     2: "-",
#     3: "x",
#     4: "|",
#     5: "--",
#     6: "||"
# }

# BLACK = (0, 0, 0)
# ROI = 200
# THICK = 18

# INTERVAL_SEC = 5
# ON_SEC = 1

# def get_vertical_left_pos(h, index):
#     x = int(0.12 * W)                  # left side
#     y_start = int(0.2 * h)             # start a bit down
#     gap = ROI + 30
#     y = y_start + index * gap
#     return y, x

# def draw_symbol(sym):
#     img = np.zeros((ROI, ROI, 3), dtype=np.uint8)
#     img[:] = 255  # white background

#     c = ROI // 2
#     m = 30

#     if sym == "|":
#         cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
#     elif sym == "||":
#         cv2.line(img,(c-25,m),(c-25,ROI-m),BLACK,THICK)
#         cv2.line(img,(c+25,m),(c+25,ROI-m),BLACK,THICK)
#     elif sym == "-":
#         cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
#     elif sym == "--":
#         cv2.line(img,(m,c-20),(ROI-m,c-20),BLACK,THICK)
#         cv2.line(img,(m,c+20),(ROI-m,c+20),BLACK,THICK)
#     elif sym == "+":
#         cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
#         cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
#     elif sym == "x":
#         cv2.line(img,(m,m),(ROI-m,ROI-m),BLACK,THICK)
#         cv2.line(img,(ROI-m,m),(m,ROI-m),BLACK,THICK)

#     return img

# cap = cv2.VideoCapture(INPUT_VIDEO)
# fps = cap.get(cv2.CAP_PROP_FPS) or 25
# W = int(cap.get(3))
# H = int(cap.get(4))

# out = cv2.VideoWriter(
#     OUTPUT_VIDEO,
#     cv2.VideoWriter_fourcc(*"mp4v"),
#     fps,
#     (W, H)
# )

# interval_f = int(INTERVAL_SEC * fps)
# on_f = int(ON_SEC * fps)
# f = 0

# while cap.isOpened():
#     ret, frame = cap.read()
#     if not ret:
#         break

#     slot = (f // interval_f) % len(MESSAGE)

#     if f % interval_f < on_f:
#         digit = MESSAGE[slot]
#         sym = DIGIT_TO_SYMBOL[digit]
#         block = draw_symbol(sym)

#         cy, cx = get_vertical_left_pos(H, slot)
#         y, x = cy - ROI//2, cx - ROI//2

#         # Check if the block fits within the frame before attempting to draw
#         if (y >= 0 and y + ROI <= H and
#             x >= 0 and x + ROI <= W):
#             mask = block[:,:,0] < 250   # black pixels
#             frame[y:y+ROI, x:x+ROI][mask] = block[mask]

#     out.write(frame)
#     f += 1

# cap.release()
# out.release()

# print("Encoded message:", MESSAGE)
# print("Saved to:", OUTPUT_VIDEO)


import cv2
import numpy as np

INPUT_VIDEO = "/content/4ktest.mp4"
OUTPUT_VIDEO = "/content/encoded_15452654651354.mp4"

# CHANGE MESSAGE HERE ONLY
MESSAGE = [1,5,4,5,2,6,5,4,6,5,1,3,5,4]   # example: 43515512

DIGIT_TO_SYMBOL = {
    1: "+",
    2: "-",
    3: "x",
    4: "|",
    5: "--",
    6: "||"
}

BLACK = (0, 0, 0)
ROI = 200
THICK = 18

INTERVAL_SEC = 5
ON_SEC = 1

def get_vertical_left_pos(h, index):
    x = int(0.12 * W)
    y_start = int(0.2 * h)
    gap = ROI + 30

    max_slots = (h - y_start - ROI) // gap
    safe_index = index % max_slots

    y = y_start + safe_index * gap
    return y, x


def draw_symbol(sym):
    img = np.zeros((ROI, ROI, 3), dtype=np.uint8)
    img[:] = 255  # white background

    c = ROI // 2
    m = 30

    if sym == "|":
        cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
    elif sym == "||":
        cv2.line(img,(c-25,m),(c-25,ROI-m),BLACK,THICK)
        cv2.line(img,(c+25,m),(c+25,ROI-m),BLACK,THICK)
    elif sym == "-":
        cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
    elif sym == "--":
        cv2.line(img,(m,c-20),(ROI-m,c-20),BLACK,THICK)
        cv2.line(img,(m,c+20),(ROI-m,c+20),BLACK,THICK)
    elif sym == "+":
        cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
        cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
    elif sym == "x":
        cv2.line(img,(m,m),(ROI-m,ROI-m),BLACK,THICK)
        cv2.line(img,(ROI-m,m),(m,ROI-m),BLACK,THICK)

    return img

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(3))
H = int(cap.get(4))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

interval_f = int(INTERVAL_SEC * fps)
on_f = int(ON_SEC * fps)
f = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    slot = (f // interval_f) % len(MESSAGE)

    if f % interval_f < on_f:
        digit = MESSAGE[slot]
        sym = DIGIT_TO_SYMBOL[digit]
        block = draw_symbol(sym)

        cy, cx = get_vertical_left_pos(H, slot)
        y, x = cy - ROI//2, cx - ROI//2

        mask = block[:,:,0] < 250   # black pixels
        frame[y:y+ROI, x:x+ROI][mask] = block[mask]

    out.write(frame)
    f += 1

cap.release()
out.release()

print("Encoded message:", MESSAGE)
print("Saved to:", OUTPUT_VIDEO)



Encoded message: [1, 5, 4, 5, 2, 6, 5, 4, 6, 5, 1, 3, 5, 4]
Saved to: /content/encoded_15452654651354.mp4


In [ ]:
import cv2
import numpy as np
from collections import Counter

VIDEO = "/content/recorded_encoded_15452654651354.mp4"

EXPECTED_MESSAGE = [1,5,4,5,2,6,5,4,6,5,1,3,5,4]

DIGIT_TO_SYMBOL = {
    1: "+",
    2: "-",
    3: "x",
    4: "|",
    5: "--",
    6: "||"
}

EXPECTED_OPS = set(DIGIT_TO_SYMBOL[d] for d in EXPECTED_MESSAGE)

# BLACK COLOR RANGE (HSV)
LOW_BLACK  = np.array([0, 0, 0])
HIGH_BLACK = np.array([180, 255, 60])

def classify_operator(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    lines = cv2.HoughLines(edges, 1, np.pi/180, 80)
    if lines is None:
        return None

    angles = [l[0][1] for l in lines]
    vert  = sum(abs(a - np.pi/2) < 0.12 for a in angles)
    horiz = sum(abs(a) < 0.12 for a in angles)
    diag  = sum(abs(a - np.pi/4) < 0.15 or abs(a - 3*np.pi/4) < 0.15 for a in angles)

    if vert and horiz:
        return "+"
    if vert >= 2:
        return "||"
    if horiz >= 2:
        return "--"
    if diag >= 2:
        return "x"
    if vert:
        return "|"
    if horiz:
        return "-"
    return None

cap = cv2.VideoCapture(VIDEO)
detected_ops = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOW_BLACK, HIGH_BLACK)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5),np.uint8))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        if w*h < 1200:
            continue

        roi = frame[y:y+h, x:x+w]
        op = classify_operator(roi)
        if op:
            detected_ops.append(op)

cap.release()

counts = Counter(detected_ops)
print("Detected operator counts:", counts)

missing = EXPECTED_OPS - set(counts.keys())

if not missing:
    print("✅ WATERMARK VERIFIED")
    print("Matched payload:", "".join(map(str, EXPECTED_MESSAGE)))
else:
    print("❌ WATERMARK NOT VERIFIED")
    print("Missing operators:", missing)


Detected operator counts: Counter({'||': 7675, '+': 7290, '--': 3006, '|': 2008, '-': 1876, 'x': 632})
✅ WATERMARK VERIFIED
Matched payload: 15452654651354


Below encoder decoder is for white operator at left stack

In [ ]:
import cv2
import numpy as np

INPUT_VIDEO = "/content/4ktest.mp4"
OUTPUT_VIDEO = "encoded_white_left.mp4"

# CHANGE MESSAGE HERE ONLY
MESSAGE = [1,5,4,5,2,6,5,4,6,5,1,3,5,4]   # example

DIGIT_TO_SYMBOL = {
    1: "+",
    2: "-",
    3: "x",
    4: "|",
    5: "--",
    6: "||"
}

WHITE = (255, 255, 255)
ROI = 200
THICK = 18

INTERVAL_SEC = 5
ON_SEC = 1

cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError("Input video not opened")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

def get_vertical_left_pos(index):
    x = int(0.12 * W)
    y_start = int(0.2 * H)
    gap = ROI + 30
    max_slots = max(1, (H - y_start - ROI) // gap)
    safe_idx = index % max_slots
    y = y_start + safe_idx * gap
    return y, x

def draw_symbol(sym):
    img = np.zeros((ROI, ROI, 3), dtype=np.uint8)  # black background
    c = ROI // 2
    m = 30

    if sym == "|":
        cv2.line(img,(c,m),(c,ROI-m),WHITE,THICK)
    elif sym == "||":
        cv2.line(img,(c-25,m),(c-25,ROI-m),WHITE,THICK)
        cv2.line(img,(c+25,m),(c+25,ROI-m),WHITE,THICK)
    elif sym == "-":
        cv2.line(img,(m,c),(ROI-m,c),WHITE,THICK)
    elif sym == "--":
        cv2.line(img,(m,c-20),(ROI-m,c-20),WHITE,THICK)
        cv2.line(img,(m,c+20),(ROI-m,c+20),WHITE,THICK)
    elif sym == "+":
        cv2.line(img,(c,m),(c,ROI-m),WHITE,THICK)
        cv2.line(img,(m,c),(ROI-m,c),WHITE,THICK)
    elif sym == "x":
        cv2.line(img,(m,m),(ROI-m,ROI-m),WHITE,THICK)
        cv2.line(img,(ROI-m,m),(m,ROI-m),WHITE,THICK)

    return img

interval_f = int(INTERVAL_SEC * fps)
on_f = int(ON_SEC * fps)
f = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    slot = (f // interval_f) % len(MESSAGE)

    if f % interval_f < on_f:
        digit = MESSAGE[slot]
        sym = DIGIT_TO_SYMBOL[digit]
        block = draw_symbol(sym)

        cy, cx = get_vertical_left_pos(slot)
        y, x = cy - ROI//2, cx - ROI//2

        if y < 0 or x < 0 or y+ROI > H or x+ROI > W:
            out.write(frame)
            f += 1
            continue

        mask = block[:,:,0] > 200   # white pixels
        frame[y:y+ROI, x:x+ROI][mask] = block[mask]

    out.write(frame)
    f += 1

cap.release()
out.release()

print("Encoded message:", MESSAGE)
print("Saved to:", OUTPUT_VIDEO)


Encoded message: [1, 5, 4, 5, 2, 6, 5, 4, 6, 5, 1, 3, 5, 4]
Saved to: encoded_white_left.mp4


In [ ]:
import cv2
import numpy as np
from collections import Counter

VIDEO = "/content/recorded_W_encoded_15452654651354.mp4"

EXPECTED_MESSAGE = [1,5,4,5,2,6,5,4,6,5,1,3,5,4]

DIGIT_TO_SYMBOL = {
    1: "+",
    2: "-",
    3: "x",
    4: "|",
    5: "--",
    6: "||"
}

EXPECTED_OPS = set(DIGIT_TO_SYMBOL[d] for d in EXPECTED_MESSAGE)

# WHITE detection (HSV)
LOW_WHITE  = np.array([0, 0, 200])
HIGH_WHITE = np.array([180, 40, 255])

def classify_operator(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)

    lines = cv2.HoughLines(edges, 1, np.pi/180, 80)
    if lines is None:
        return None

    angles = [l[0][1] for l in lines]
    vert  = sum(abs(a - np.pi/2) < 0.12 for a in angles)
    horiz = sum(abs(a) < 0.12 for a in angles)
    diag  = sum(abs(a - np.pi/4) < 0.15 or abs(a - 3*np.pi/4) < 0.15 for a in angles)

    if vert and horiz:
        return "+"
    if vert >= 2:
        return "||"
    if horiz >= 2:
        return "--"
    if diag >= 2:
        return "x"
    if vert:
        return "|"
    if horiz:
        return "-"
    return None

cap = cv2.VideoCapture(VIDEO)
detected_ops = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOW_WHITE, HIGH_WHITE)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5,5),np.uint8))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        if w*h < 1200:
            continue

        roi = frame[y:y+h, x:x+w]
        op = classify_operator(roi)
        if op:
            detected_ops.append(op)

cap.release()

counts = Counter(detected_ops)
print("Detected operator counts:", counts)

missing = EXPECTED_OPS - set(counts.keys())

if not missing:
    print("✅ WATERMARK VERIFIED")
    print("Matched payload:", "".join(map(str, EXPECTED_MESSAGE)))
else:
    print("❌ WATERMARK NOT VERIFIED")
    print("Missing operators:", missing)


Detected operator counts: Counter({'||': 3121, '+': 1682, '|': 1490, '--': 1427, '-': 960, 'x': 114})
✅ WATERMARK VERIFIED
Matched payload: 15452654651354


Below encoder decoder is for right black vertical line/stack

In [ ]:
import cv2
import numpy as np

INPUT_VIDEO = "/content/4ktest.mp4"
OUTPUT_VIDEO = "/content/encoded_black_right.mp4"

MESSAGE = [5,4,6,5,6,1,5,4,6,5,1,3,5,5,2]

DIGIT_TO_SYMBOL = {
    1:"+", 2:"-", 3:"x", 4:"|", 5:"--", 6:"||"
}

BLACK = (0,0,0)
ROI = 200
THICK = 18
INTERVAL_SEC = 5
ON_SEC = 1

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(3)); H = int(cap.get(4))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,(W,H)
)

def get_right_vertical_pos(idx):
    x = int(0.88 * W)
    y0 = int(0.2 * H)
    gap = ROI + 30
    max_slots = max(1,(H-y0-ROI)//gap)
    idx = idx % max_slots
    return y0 + idx*gap, x

def draw(sym):
    img = np.ones((ROI,ROI,3),np.uint8)*255
    c,m = ROI//2,30
    if sym=="|": cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
    elif sym=="||":
        cv2.line(img,(c-25,m),(c-25,ROI-m),BLACK,THICK)
        cv2.line(img,(c+25,m),(c+25,ROI-m),BLACK,THICK)
    elif sym=="-": cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
    elif sym=="--":
        cv2.line(img,(m,c-20),(ROI-m,c-20),BLACK,THICK)
        cv2.line(img,(m,c+20),(ROI-m,c+20),BLACK,THICK)
    elif sym=="+":
        cv2.line(img,(c,m),(c,ROI-m),BLACK,THICK)
        cv2.line(img,(m,c),(ROI-m,c),BLACK,THICK)
    elif sym=="x":
        cv2.line(img,(m,m),(ROI-m,ROI-m),BLACK,THICK)
        cv2.line(img,(ROI-m,m),(m,ROI-m),BLACK,THICK)
    return img

interval_f = int(INTERVAL_SEC*fps)
on_f = int(ON_SEC*fps)
f=0

while cap.isOpened():
    ret,frame = cap.read()
    if not ret: break

    slot = (f//interval_f) % len(MESSAGE)
    if f%interval_f < on_f:
        sym = DIGIT_TO_SYMBOL[MESSAGE[slot]]
        blk = draw(sym)
        cy,cx = get_right_vertical_pos(slot)
        y,x = cy-ROI//2, cx-ROI//2
        if 0<=y and 0<=x and y+ROI<=H and x+ROI<=W:
            mask = blk[:,:,0]<250
            frame[y:y+ROI,x:x+ROI][mask] = blk[mask]

    out.write(frame)
    f+=1

cap.release(); out.release()
print("BLACK encoded:", MESSAGE)


BLACK encoded: [5, 4, 6, 5, 6, 1, 5, 4, 6, 5, 1, 3, 5, 5, 2]


In [ ]:
import cv2, numpy as np
from collections import Counter

VIDEO = "/content/VID20260112181007.mp4"
EXPECTED = [5,4,6,5,6,1,5,4,6,5,1,3,5,5,2]

DIGIT_TO_SYMBOL = {1:"+",2:"-",3:"x",4:"|",5:"--",6:"||"}
EXPECTED_OPS = set(DIGIT_TO_SYMBOL[d] for d in EXPECTED)

LOW_BLACK = np.array([0,0,0])
HIGH_BLACK = np.array([180,255,70])

def classify(roi):
    g=cv2.cvtColor(roi,cv2.COLOR_BGR2GRAY)
    e=cv2.Canny(g,50,150)
    l=cv2.HoughLines(e,1,np.pi/180,80)
    if l is None: return None
    a=[x[0][1] for x in l]
    v=sum(abs(x-np.pi/2)<0.12 for x in a)
    h=sum(abs(x)<0.12 for x in a)
    d=sum(abs(x-np.pi/4)<0.15 or abs(x-3*np.pi/4)<0.15 for x in a)
    if v and h: return "+"
    if v>=2: return "||"
    if h>=2: return "--"
    if d>=2: return "x"
    if v: return "|"
    if h: return "-"
    return None

cap=cv2.VideoCapture(VIDEO)
ops=[]
while cap.isOpened():
    r,f=cap.read()
    if not r: break
    hsv=cv2.cvtColor(f,cv2.COLOR_BGR2HSV)
    m=cv2.inRange(hsv,LOW_BLACK,HIGH_BLACK)
    cnts,_=cv2.findContours(m,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    for c in cnts:
        x,y,w,h=cv2.boundingRect(c)
        if w*h<1200: continue
        o=classify(f[y:y+h,x:x+w])
        if o: ops.append(o)
cap.release()

cnt=Counter(ops)
print(cnt)
print("VERIFIED" if EXPECTED_OPS<=cnt.keys() else "FAILED")


Counter({'||': 11392, '+': 5656, '|': 2770, '-': 1505, '--': 1369, 'x': 450})
VERIFIED


Below is encoder decoder for white right vertical line/stack

In [ ]:
import cv2
import numpy as np
import subprocess
import os

INPUT_VIDEO = "/content/4ktest.mp4"
TEMP_VIDEO  = "/content/encoded_white_right_no_audio.mp4"
OUTPUT_VIDEO = "/content/encoded_white_right.mp4"

MESSAGE = [4,5,6,1,6,2,5,4,6,5,1,2,3,3,5]

DIGIT_TO_SYMBOL = {
    1:"+", 2:"-", 3:"x", 4:"|", 5:"--", 6:"||"
}

WHITE = (255,255,255)
ROI = 200
THICK = 18
INTERVAL_SEC = 5
ON_SEC = 1

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    TEMP_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W,H)
)

def get_right_vertical_pos(idx):
    x = int(0.88 * W)
    y0 = int(0.2 * H)
    gap = ROI + 30
    max_slots = max(1, (H - y0 - ROI) // gap)
    idx = idx % max_slots
    return y0 + idx * gap, x

def draw(sym):
    img = np.zeros((ROI,ROI,3), np.uint8)  # BLACK background
    c, m = ROI//2, 30

    if sym == "|":
        cv2.line(img,(c,m),(c,ROI-m),WHITE,THICK)
    elif sym == "||":
        cv2.line(img,(c-25,m),(c-25,ROI-m),WHITE,THICK)
        cv2.line(img,(c+25,m),(c+25,ROI-m),WHITE,THICK)
    elif sym == "-":
        cv2.line(img,(m,c),(ROI-m,c),WHITE,THICK)
    elif sym == "--":
        cv2.line(img,(m,c-20),(ROI-m,c-20),WHITE,THICK)
        cv2.line(img,(m,c+20),(ROI-m,c+20),WHITE,THICK)
    elif sym == "+":
        cv2.line(img,(c,m),(c,ROI-m),WHITE,THICK)
        cv2.line(img,(m,c),(ROI-m,c),WHITE,THICK)
    elif sym == "x":
        cv2.line(img,(m,m),(ROI-m,ROI-m),WHITE,THICK)
        cv2.line(img,(ROI-m,m),(m,ROI-m),WHITE,THICK)

    return img

interval_f = int(INTERVAL_SEC * fps)
on_f = int(ON_SEC * fps)
f = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    slot = (f // interval_f) % len(MESSAGE)

    if f % interval_f < on_f:
        sym = DIGIT_TO_SYMBOL[MESSAGE[slot]]
        blk = draw(sym)

        cy, cx = get_right_vertical_pos(slot)
        y, x = cy - ROI//2, cx - ROI//2

        if 0 <= y and 0 <= x and y+ROI <= H and x+ROI <= W:
            mask = blk[:,:,0] > 200   # WHITE pixels only
            frame[y:y+ROI, x:x+ROI][mask] = blk[mask]

    out.write(frame)
    f += 1

cap.release()
out.release()

# ================= AUDIO RESTORE =================

subprocess.run([
    "ffmpeg", "-y",
    "-i", TEMP_VIDEO,
    "-i", INPUT_VIDEO,
    "-map", "0:v:0",
    "-map", "1:a:0",
    "-c:v", "copy",
    "-c:a", "copy",
    OUTPUT_VIDEO
], check=True)


print("✅ WHITE encoded with audio:", MESSAGE)
print("Saved to:", OUTPUT_VIDEO)


✅ WHITE encoded with audio: [4, 5, 6, 1, 6, 2, 5, 4, 6, 5, 1, 2, 3, 3, 5]
Saved to: /content/encoded_white_right.mp4


In [ ]:
import cv2, numpy as np
from collections import Counter

VIDEO = "/content/recorded_encoded_white_right.mp4"

EXPECTED = [4,5,6,1,6,2,5,4,6,5,1,2,3,3,5]

DIGIT_TO_SYMBOL = {
    1:"+", 2:"-", 3:"x", 4:"|", 5:"--", 6:"||"
}
EXPECTED_OPS = set(DIGIT_TO_SYMBOL[d] for d in EXPECTED)

# WHITE detection (camera-safe)
LOW_WHITE  = np.array([0, 0, 200])
HIGH_WHITE = np.array([180, 40, 255])

def classify(roi):
    g = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    e = cv2.Canny(g, 50, 150)
    l = cv2.HoughLines(e, 1, np.pi/180, 80)
    if l is None:
        return None

    a = [x[0][1] for x in l]
    v = sum(abs(x - np.pi/2) < 0.12 for x in a)
    h = sum(abs(x) < 0.12 for x in a)
    d = sum(abs(x - np.pi/4) < 0.15 or abs(x - 3*np.pi/4) < 0.15 for x in a)

    if v and h: return "+"
    if v >= 2:   return "||"
    if h >= 2:   return "--"
    if d >= 2:   return "x"
    if v:        return "|"
    if h:        return "-"
    return None

cap = cv2.VideoCapture(VIDEO)
ops = []

while cap.isOpened():
    r, f = cap.read()
    if not r:
        break

    hsv = cv2.cvtColor(f, cv2.COLOR_BGR2HSV)
    m = cv2.inRange(hsv, LOW_WHITE, HIGH_WHITE)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((5,5),np.uint8))

    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for c in cnts:
        x,y,w,h = cv2.boundingRect(c)
        if w*h < 1200:
            continue
        o = classify(f[y:y+h, x:x+w])
        if o:
            ops.append(o)

cap.release()

cnt = Counter(ops)
print("Detected operator counts:", cnt)

missing = EXPECTED_OPS - set(cnt.keys())

if not missing:
    print("✅ WATERMARK VERIFIED")
    print("Matched payload:", "".join(map(str, EXPECTED)))
else:
    print("❌ WATERMARK NOT VERIFIED")
    print("Missing operators:", missing)


Detected operator counts: Counter({'||': 1095, '+': 644, '|': 602, '--': 398, '-': 390, 'x': 96})
✅ WATERMARK VERIFIED
Matched payload: 456162546512335


Now we have to reduce the size of the operators. We'll go with white operators on right side only

In [ ]:
import cv2
import numpy as np
import subprocess
import os

INPUT_VIDEO = "/content/4ktest.mp4"
TEMP_VIDEO  = "/content/encoded_white_right_small_no_audio.mp4"
OUTPUT_VIDEO = "/content/encoded_white_right_small.mp4"

MESSAGE = [2,5,6,1,5,6,5,1,2,3,4]

DIGIT_TO_SYMBOL = {
    1:"+", 2:"-", 3:"x", 4:"|", 5:"--", 6:"||"
}

WHITE = (255,255,255)

ROI = 100          # 🔽 half size
THICK = 9          # 🔽 half thickness
INTERVAL_SEC = 5
ON_SEC = 1

cap = cv2.VideoCapture(INPUT_VIDEO)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps <= 1:
    fps = 25

W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    TEMP_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H)
)

if not out.isOpened():
    raise RuntimeError("VideoWriter failed to open")


def get_right_vertical_pos(idx):
    x = int(0.88 * W)
    y0 = int(0.2 * H)
    gap = ROI + 20
    max_slots = max(1, (H - y0 - ROI) // gap)
    idx = idx % max_slots
    return y0 + idx * gap, x

def draw(sym):
    img = np.zeros((ROI, ROI, 3), np.uint8)
    c, m = ROI // 2, 15

    if sym == "|":
        cv2.line(img, (c,m), (c,ROI-m), WHITE, THICK)
    elif sym == "||":
        cv2.line(img, (c-12,m), (c-12,ROI-m), WHITE, THICK)
        cv2.line(img, (c+12,m), (c+12,ROI-m), WHITE, THICK)
    elif sym == "-":
        cv2.line(img, (m,c), (ROI-m,c), WHITE, THICK)
    elif sym == "--":
        cv2.line(img, (m,c-10), (ROI-m,c-10), WHITE, THICK)
        cv2.line(img, (m,c+10), (ROI-m,c+10), WHITE, THICK)
    elif sym == "+":
        cv2.line(img, (c,m), (c,ROI-m), WHITE, THICK)
        cv2.line(img, (m,c), (ROI-m,c), WHITE, THICK)
    elif sym == "x":
        cv2.line(img, (m,m), (ROI-m,ROI-m), WHITE, THICK)
        cv2.line(img, (ROI-m,m), (m,ROI-m), WHITE, THICK)

    return img

interval_f = int(INTERVAL_SEC * fps)
on_f = int(ON_SEC * fps)
f = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    slot = (f // interval_f) % len(MESSAGE)

    if f % interval_f < on_f:
        sym = DIGIT_TO_SYMBOL[MESSAGE[slot]]
        blk = draw(sym)

        cy, cx = get_right_vertical_pos(slot)
        y, x = cy - ROI//2, cx - ROI//2

        if 0 <= y and 0 <= x and y+ROI <= H and x+ROI <= W:
            mask = blk[:,:,0] > 200
            frame[y:y+ROI, x:x+ROI][mask] = blk[mask]

    out.write(frame)
    f += 1

cap.release()
out.release()

# ╪ Restore original audio (NO trimming)
# Check if the input video has an audio track
try:
    # Use ffprobe to check for an audio stream
    audio_check_cmd = [
        "ffprobe", "-v", "error", "-select_streams", "a",
        "-show_entries", "stream=codec_type", "-of", "default=noprint_wrappers=1:nokey=1",
        INPUT_VIDEO
    ]
    audio_probe_result = subprocess.run(audio_check_cmd, capture_output=True, text=True, check=False)

    has_audio = "audio" in audio_probe_result.stdout.strip()
except FileNotFoundError:
    print("ffprobe not found. Cannot check for audio stream. Proceeding without audio mapping.")
    has_audio = False
except Exception as e:
    print(f"Error checking for audio stream with ffprobe: {e}. Proceeding without audio mapping.")
    has_audio = False

ffmpeg_command = [
    "ffmpeg", "-y",
    "-i", TEMP_VIDEO,
    "-i", INPUT_VIDEO,
    "-map", "0:v:0",
    "-c:v", "copy"
]

if has_audio:
    ffmpeg_command.extend(["-map", "1:a:0", "-c:a", "copy"])
else:
    print("No audio stream detected in the input video. Output video will not have audio.")

ffmpeg_command.append(OUTPUT_VIDEO)

subprocess.run(ffmpeg_command, check=True)

print("✅ White SMALL encoded with audio:", MESSAGE)


✅ White SMALL encoded with audio: [2, 5, 6, 1, 5, 6, 5, 1, 2, 3, 4]


In [ ]:
import cv2
import numpy as np
from collections import Counter

VIDEO = "/content/recorded_encoded_white_right_small.mp4"

EXPECTED_MESSAGE = [2,5,6,1,5,6,5,1,2,3,4]

DIGIT_TO_SYMBOL = {
    1:"+", 2:"-", 3:"x", 4:"|", 5:"--", 6:"||"
}

EXPECTED_OPS = set(DIGIT_TO_SYMBOL[d] for d in EXPECTED_MESSAGE)

LOW_WHITE  = np.array([0, 0, 200])
HIGH_WHITE = np.array([180, 40, 255])

def classify_operator(roi):
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 40, 120)

    lines = cv2.HoughLines(edges, 1, np.pi/180, 50)
    if lines is None:
        return None

    angles = [l[0][1] for l in lines]
    vert  = sum(abs(a - np.pi/2) < 0.12 for a in angles)
    horiz = sum(abs(a) < 0.12 for a in angles)
    diag  = sum(abs(a - np.pi/4) < 0.15 or abs(a - 3*np.pi/4) < 0.15 for a in angles)

    if vert and horiz: return "+"
    if vert >= 2: return "||"
    if horiz >= 2: return "--"
    if diag >= 2: return "x"
    if vert: return "|"
    if horiz: return "-"
    return None

cap = cv2.VideoCapture(VIDEO)
detected_ops = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOW_WHITE, HIGH_WHITE)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3),np.uint8))

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for cnt in contours:
        x,y,w,h = cv2.boundingRect(cnt)
        if w*h < 300:   # 🔽 reduced threshold
            continue

        roi = frame[y:y+h, x:x+w]
        op = classify_operator(roi)
        if op:
            detected_ops.append(op)

cap.release()

counts = Counter(detected_ops)
print("Detected operator counts:", counts)

missing = EXPECTED_OPS - set(counts.keys())

if not missing:
    print("✅ WATERMARK VERIFIED")
    print("Matched payload:", "".join(map(str, EXPECTED_MESSAGE)))
else:
    print("❌ WATERMARK NOT VERIFIED")
    print("Missing operators:", missing)


Detected operator counts: Counter({'||': 2968, '+': 1942, '--': 1492, '|': 1195, '-': 892, 'x': 240})
✅ WATERMARK VERIFIED
Matched payload: 25615651234
